# Fixed Income Pricing Engine Demo

This notebook walks through bond construction, curve bootstrapping, pricing, analytics, portfolio aggregation, hedge sizing, and visualization.

In [ ]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fixed_income.analytics.convexity import effective_convexity
from fixed_income.analytics.duration import effective_duration, price_from_curve
from fixed_income.analytics.dv01 import dv01_from_curve
from fixed_income.analytics.key_rate_dv01 import key_rate_dv01
from fixed_income.bonds.bond import FixedRateBond
from fixed_income.curves.bootstrapper import Bootstrapper, MarketInstrument
from fixed_income.portfolio.portfolio import Portfolio
from fixed_income.visualization.plots import (
    plot_cash_flows,
    plot_key_rate_dv01,
    plot_portfolio_risk,
    plot_price_yield,
    plot_yield_curve,
)


## 1. Create a 10Y 5% semi-annual Treasury-style bond

In [ ]:
settlement_date = date(2025, 1, 1)
bond = FixedRateBond(
    face_value=1000.0,
    coupon_rate=0.05,
    issue_date=settlement_date,
    maturity_date=date(2035, 1, 1),
    frequency=2,
    day_count_convention="ACT/ACT",
)
bond.get_cash_flows().head()

## 2. Bootstrap a yield curve from sample deposit and swap quotes

In [ ]:
market_instruments = [
    MarketInstrument("deposit", "1M", 0.0500, 2),
    MarketInstrument("deposit", "3M", 0.0510, 2),
    MarketInstrument("deposit", "6M", 0.0515, 2),
    MarketInstrument("deposit", "1Y", 0.0520, 2),
    MarketInstrument("swap", "2Y", 0.0525, 2),
    MarketInstrument("swap", "5Y", 0.0530, 2),
    MarketInstrument("swap", "10Y", 0.0535, 2),
    MarketInstrument("swap", "30Y", 0.0540, 2),
]
zero_curve = Bootstrapper(market_instruments).bootstrap()
pd.DataFrame({
    "tenor_years": zero_curve.maturities,
    "discount_factor": zero_curve.discount_factors,
    "spot_rate": [zero_curve.spot_rate(float(t)) for t in zero_curve.maturities],
})

## 3. Price the bond on the curve

In [ ]:
curve_price = price_from_curve(bond, zero_curve, settlement_date)
curve_price

## 4. Compute Duration, DV01, and Convexity

In [ ]:
analytics_summary = pd.DataFrame(
    {
        "metric": ["effective_duration", "curve_dv01", "effective_convexity"],
        "value": [
            effective_duration(bond, zero_curve, settlement_date),
            dv01_from_curve(bond, zero_curve, settlement_date),
            effective_convexity(bond, zero_curve, settlement_date),
        ],
    }
)
analytics_summary

## 5. Compute the Key Rate DV01 profile

In [ ]:
key_rate_map, key_rate_series = key_rate_dv01(bond, zero_curve, settlement_date)
key_rate_series.to_frame(name="dv01")

## 6. Build a 3-bond portfolio and generate a risk report

In [ ]:
portfolio = Portfolio()
portfolio.add_position(bond=bond, notional=2_000_000.0, direction=1)
portfolio.add_position(
    bond=FixedRateBond(
        face_value=1000.0,
        coupon_rate=0.04,
        issue_date=settlement_date,
        maturity_date=date(2029, 1, 1),
        frequency=2,
        day_count_convention="ACT/ACT",
    ),
    notional=1_000_000.0,
    direction=-1,
)
hedge_bond = FixedRateBond(
    face_value=1000.0,
    coupon_rate=0.06,
    issue_date=settlement_date,
    maturity_date=date(2038, 1, 1),
    frequency=2,
    day_count_convention="ACT/ACT",
)
portfolio.add_position(bond=hedge_bond, notional=1_500_000.0, direction=1)
display(portfolio.risk_report(zero_curve, settlement_date))
display(portfolio.key_rate_dv01_profile(zero_curve, settlement_date))

## 7. Compute the hedge trade needed to flatten DV01 to zero

In [ ]:
hedge_notional = portfolio.hedge_trade(0.0, hedge_bond, zero_curve, settlement_date)
hedge_notional

## 8. Show all five visualizations

In [ ]:
plot_yield_curve(zero_curve).show()
plot_price_yield(bond, settlement_date=settlement_date).show()
plot_key_rate_dv01(key_rate_series).show()
plot_cash_flows(bond, settlement_date=settlement_date).show()
plot_portfolio_risk(portfolio, zero_curve, settlement_date).show()